# LIAR — SHAP and LIME Explainability

Applies SHAP and LIME to the LIAR-trained DistilBERT's predictions - find where the TF-IDF baseline and DistilBERT disagree on predictions, then use SHAP/LIME to understand what's driving each model's decision on those cases.

Requires: `pip install shap lime`

## 1. Load LIAR-Trained DistilBERT and Test Data


In [1]:
import pandas as pd
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

test_df = pd.read_csv("liar_test.csv")
train_df = pd.read_csv("liar_train.csv")

tokenizer = DistilBertTokenizerFast.from_pretrained("./distilbert_liar_final")
model = DistilBertForSequenceClassification.from_pretrained("./distilbert_liar_final")
model.to(device)
model.eval()

MAX_LENGTH = 64  # matches the LIAR training setup

print("Test set size:", len(test_df))
test_df.head()


Using device: cuda


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Test set size: 790


,text,label,label_id
0,Building a wall on the U.S.-Mexico border will...,real,0
1,Wisconsin is on pace to double the number of l...,fake,1
2,Says John McCain has done nothing to help the ...,fake,1
3,When asked by a reporter whether hes at the ce...,fake,1
4,Over the past five years the federal governmen...,real,0


In [2]:
import numpy as np

def distilbert_predict_proba(texts, batch_size=8):
    all_probs = []
    texts = list(texts)
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True, max_length=MAX_LENGTH, return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
        all_probs.append(probs)
        del inputs, logits
        torch.cuda.empty_cache()
    return np.vstack(all_probs)

all_probs = distilbert_predict_proba(test_df["text"].astype(str).tolist())
test_df["pred"] = all_probs.argmax(axis=1)
test_df["confidence"] = all_probs.max(axis=1)

print(f"Test accuracy: {(test_df['pred'] == test_df['label_id']).mean():.4f}")


Test accuracy: 0.6886


## 2. Baseline Model + Prediction Disagreements

Finds cases where the baseline and DistilBERT disagree — including specifically false positives (real statements wrongly predicted as fake).

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

baseline_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=20000, ngram_range=(1, 2), stop_words="english", min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
baseline_pipeline.fit(train_df["text"], train_df["label_id"])

baseline_preds = baseline_pipeline.predict(test_df["text"].astype(str))
test_df["baseline_pred"] = baseline_preds
test_df["disagreement"] = test_df["baseline_pred"] != test_df["pred"]

print(f"Total disagreements: {test_df['disagreement'].sum()} out of {len(test_df)} ({test_df['disagreement'].mean():.1%})")

disagreement_df = test_df[test_df["disagreement"]].copy()
disagreement_df[["text", "label", "baseline_pred", "pred"]].head(10)


Total disagreements: 209 out of 790 (26.5%)


,text,label,baseline_pred,pred
1,Wisconsin is on pace to double the number of l...,fake,0,1
13,There have not been any public safety issues i...,real,0,1
14,The number of illegal immigrants could be 3 mi...,fake,0,1
17,"Now, there was a time when someone like Scalia...",real,1,0
20,Its been since 1888 that a Senate of a differe...,fake,1,0
21,"Under Rosemary Lehmberg, the Travis County D.A...",fake,0,1
26,Says he won the second debate with Hillary Cli...,fake,1,0
33,ACORN will be a paid partner with the Census B...,fake,0,1
38,Says Marco Rubio said Social Security and Medi...,real,1,0
44,What the facts say is ...the best scenario for...,fake,0,1


In [4]:
# False positives specifically: true label real, but predicted fake
baseline_fp = test_df[(test_df["label_id"] == 0) & (test_df["baseline_pred"] == 1)]
distilbert_fp = test_df[(test_df["label_id"] == 0) & (test_df["pred"] == 1)]

print(f"Baseline false positives: {len(baseline_fp)}")
print(f"DistilBERT false positives: {len(distilbert_fp)}")


Baseline false positives: 102
DistilBERT false positives: 133


## 3. SHAP Explanations

Uses masking-based `shap.Explainer` with a text masker.


In [5]:
import shap

masker = shap.maskers.Text(tokenizer)
explainer = shap.Explainer(distilbert_predict_proba, masker)

def get_top_shap_tokens(text, class_idx, top_k=5):
    shap_values = explainer([text])
    tokens = shap_values.data[0]
    values = shap_values.values[0, :, class_idx]
    pairs = list(zip(tokens, values))
    pairs.sort(key=lambda x: abs(x[1]), reverse=True)
    return pairs[:top_k]

example = disagreement_df.iloc[0] if len(disagreement_df) > 0 else test_df.iloc[0]
top_tokens = get_top_shap_tokens(example["text"], class_idx=int(example["pred"]), top_k=5)

print("Text:", example["text"])
print("True label:", example["label"], "| Predicted:", "fake" if example["pred"] == 1 else "real")
print("\nTop SHAP tokens:")
for token, val in top_tokens:
    direction = "toward FAKE" if val > 0 else "toward REAL"
    print(f"  '{token.strip()}': {val:.4f} ({direction})")


Text: Wisconsin is on pace to double the number of layoffs this year.
True label: fake | Predicted: fake

Top SHAP tokens:
  'Wisconsin': 0.0696 (toward FAKE)
  'double': -0.0315 (toward REAL)
  'pace': -0.0266 (toward REAL)
  'year': -0.0258 (toward REAL)
  'lay': 0.0250 (toward FAKE)


## 4. LIME Explanations


In [6]:
from lime.lime_text import LimeTextExplainer
import time

lime_explainer = LimeTextExplainer(class_names=["real", "fake"])

def get_top_lime_tokens(text, top_k=5, num_samples=500):
    exp = lime_explainer.explain_instance(text, distilbert_predict_proba, num_features=top_k, num_samples=num_samples)
    return exp.as_list()

lime_tokens = get_top_lime_tokens(example["text"], top_k=5)

print("Text:", example["text"])
print("\nTop LIME tokens:")
for word, weight in lime_tokens:
    direction = "toward FAKE" if weight > 0 else "toward REAL"
    print(f"  '{word}': {weight:.4f} ({direction})")


Text: Wisconsin is on pace to double the number of layoffs this year.

Top LIME tokens:
  'Wisconsin': 0.2248 (toward FAKE)
  'double': -0.1483 (toward REAL)
  'year': -0.1191 (toward REAL)
  'pace': -0.0953 (toward REAL)
  'layoffs': 0.0687 (toward FAKE)


## 5. SHAP vs LIME — Timing and Word Overlap Comparison

Runs both methods across several disagreement cases, comparing generation time and word overlap between the two explanation methods.


In [7]:
comparison_rows = []
n_cases = min(5, len(disagreement_df)) if len(disagreement_df) > 0 else 5
source_df = disagreement_df if len(disagreement_df) > 0 else test_df

for idx in range(n_cases):
    row = source_df.iloc[idx]

    start = time.time()
    shap_tokens = get_top_shap_tokens(row["text"], class_idx=int(row["pred"]), top_k=5)
    shap_time = time.time() - start
    shap_words = set(t.strip().lower() for t, v in shap_tokens)

    start = time.time()
    lime_tokens = get_top_lime_tokens(row["text"], top_k=5)
    lime_time = time.time() - start
    lime_words = set(w.strip().lower() for w, v in lime_tokens)

    overlap = shap_words & lime_words
    overlap_pct = len(overlap) / max(len(shap_words | lime_words), 1)

    comparison_rows.append({
        "text_preview": row["text"][:80],
        "true_label": row["label"],
        "baseline_pred": "fake" if row["baseline_pred"] == 1 else "real",
        "distilbert_pred": "fake" if row["pred"] == 1 else "real",
        "shap_words": list(shap_words),
        "shap_time_sec": shap_time,
        "lime_words": list(lime_words),
        "lime_time_sec": lime_time,
        "word_overlap_pct": overlap_pct,
    })

shap_lime_comparison = pd.DataFrame(comparison_rows)
shap_lime_comparison.to_csv("liar_shap_lime_comparison.csv", index=False)
shap_lime_comparison


,text_preview,true_label,baseline_pred,distilbert_pred,shap_words,shap_time_sec,lime_words,lime_time_sec,word_overlap_pct
0,Wisconsin is on pace to double the number of l...,fake,real,fake,"[year, double, wisconsin, pace, lay]",0.350993,"[year, double, on, wisconsin, pace]",0.498964,0.666667
1,There have not been any public safety issues i...,real,real,fake,"[transgender, been, ., any, of]",0.701452,"[safety, transgender, been, not, there]",0.969539,0.250000
2,The number of illegal immigrants could be 3 mi...,fake,real,fake,"[number, be, the, ., illegal]",0.384251,"[3, the, 30, illegal, could]",0.512018,0.250000
3,"Now, there was a time when someone like Scalia...",real,fake,real,"[alia, votes, now, gin, ,]",0.662090,"[votes, 95, got, plus, ginsburg]",0.583227,0.111111
4,Its been since 1888 that a Senate of a differe...,fake,fake,real,"[since, a, than, house, nominee]",0.680605,"[since, than, house, nominee, president]",0.648948,0.666667
